# Cozi (Queue)

Coada este o structură **FIFO** (*First In, First Out*): primul element introdus este primul scos.

Operații, toate în **O(1)**:
- `push(x)` / `enqueue(x)` – adaugă la final
- `pop()` / `dequeue()` – scoate de la început
- `front()` – primul element
- `empty()`

⚠️ În Python, `lista.pop(0)` este **O(n)** (mută toate elementele), deci o listă simplă nu e o coadă eficientă.

## 1. Coadă circulară pe vector de capacitate fixă

Reținem indicele primului element (`head`) și numărul de elemente (`size`). Indicii „se întorc” la 0 cu `% capacity`.

In [ ]:
class CircularQueue:
    def __init__(self, capacity):
        self._data = [None] * capacity
        self.capacity = capacity
        self.head = 0
        self.size = 0

    def push(self, x):
        if self.size == self.capacity:
            raise OverflowError("coada este plină")
        tail = (self.head + self.size) % self.capacity
        self._data[tail] = x
        self.size += 1

    def pop(self):
        if self.size == 0:
            raise IndexError("pop din coadă goală")
        x = self._data[self.head]
        self._data[self.head] = None
        self.head = (self.head + 1) % self.capacity
        self.size -= 1
        return x

    def front(self):
        if self.size == 0:
            raise IndexError("coada este goală")
        return self._data[self.head]

    def empty(self):
        return self.size == 0

    def __repr__(self):
        items = [self._data[(self.head + i) % self.capacity] for i in range(self.size)]
        return f"Queue({items}) | vector intern: {self._data}, head={self.head}"


q = CircularQueue(4)
for x in [1, 2, 3]:
    q.push(x)
print(q)
print("pop:", q.pop(), q.pop())
q.push(4); q.push(5); q.push(6)   # se întoarce la începutul vectorului
print(q)

## 2. Coadă implementată cu listă înlănțuită

Scoatem de la `head`, adăugăm la `tail`.

In [ ]:
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None


class LinkedQueue:
    def __init__(self):
        self.head = None
        self.tail = None

    def push(self, x):
        node = Node(x)
        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node

    def pop(self):
        if self.head is None:
            raise IndexError("pop din coadă goală")
        x = self.head.value
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        return x

    def front(self):
        return self.head.value

    def empty(self):
        return self.head is None


q = LinkedQueue()
for c in "abc":
    q.push(c)
print([q.pop() for _ in range(3)])   # aceeași ordine

## 3. Coadă din două stive (problemă clasică de seminar)

- `push` → în stiva `inbox`
- `pop` → dacă `outbox` e goală, mutăm tot din `inbox` în `outbox` (ordinea se inversează), apoi scoatem din `outbox`

Fiecare element este mutat cel mult o dată ⇒ **O(1) amortizat** pe operație.

In [ ]:
class QueueFromTwoStacks:
    def __init__(self):
        self.inbox = []
        self.outbox = []

    def push(self, x):
        self.inbox.append(x)

    def pop(self):
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())
        if not self.outbox:
            raise IndexError("pop din coadă goală")
        return self.outbox.pop()


q = QueueFromTwoStacks()
q.push(1); q.push(2); q.push(3)
print(q.pop())          # 1
q.push(4)
print(q.pop(), q.pop(), q.pop())   # 2 3 4

## 4. În practică: `collections.deque`

`deque` (double-ended queue) are `append`, `appendleft`, `pop`, `popleft` în O(1).

### Aplicație: problema lui Josephus
`n` persoane în cerc, se elimină fiecare a `k`-a persoană. Cine rămâne ultimul?

In [ ]:
from collections import deque


def josephus(n, k):
    q = deque(range(1, n + 1))
    order = []
    while len(q) > 1:
        for _ in range(k - 1):
            q.append(q.popleft())   # trecem primii k-1 la final
        order.append(q.popleft())   # al k-lea este eliminat
    return order, q[0]


order, survivor = josephus(7, 3)
print("ordinea eliminării:", order)
print("supraviețuitor:", survivor)